In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

DATA, OUT = Path("data"), Path("dashboard")
OUT.mkdir(exist_ok=True)

b = joblib.load("models/credit_model.joblib")
model, features, cat_cols, cats = b["model"], b["features"], b["cat_cols"], b["categories"]

t = pd.read_parquet(DATA / "test_clean.parquet")
for c in cat_cols:
    t[c] = pd.Categorical(t[c], categories=cats[c])
pr = pd.read_parquet(DATA / "test_preds.parquet").set_index("SK_ID_CURR")
t["p"] = pr.loc[t["SK_ID_CURR"], "p_lgb"].values
t["band"] = np.select([t["p"] >= 0.25, t["p"] >= 0.12], ["Decline", "Review"], default="Approve")
t["risk_decile"] = pd.qcut(t["p"], 10, labels=range(1, 11)).astype(int)
t["age_group"] = pd.cut(t["age_years"], [20, 30, 40, 50, 60, 70],
                        labels=["20-29", "30-39", "40-49", "50-59", "60-69"]).astype(str)

# 1. Bands
bands = t.groupby("band").agg(applicants=("TARGET", "size"), defaults=("TARGET", "sum"),
                              loan_amount=("AMT_CREDIT", "sum")).reset_index()
bands["default_rate"] = bands["defaults"] / bands["applicants"]

# 2. Deciles (calibration)
deciles = t.groupby("risk_decile").agg(applicants=("TARGET", "size"),
                                       avg_predicted=("p", "mean"),
                                       actual_default_rate=("TARGET", "mean")).reset_index()

# 3. Segments
seg = []
for col in ["age_group", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "CODE_GENDER"]:
    g = t.groupby(col, observed=True).agg(
        applicants=("TARGET", "size"), actual_default_rate=("TARGET", "mean"),
        avg_predicted=("p", "mean"),
        declined_share=("band", lambda s: (s == "Decline").mean())).reset_index()
    g = g.rename(columns={col: "segment"})
    g.insert(0, "dimension", col)
    seg.append(g)
segments = pd.concat(seg)
segments = segments[segments["applicants"] >= 50]   # no tiny groups

# 4. Feature importance
sample = t[features].sample(5000, random_state=42)
contrib = model.predict(sample, pred_contrib=True)[:, :-1]
importance = (pd.Series(np.abs(contrib).mean(axis=0), index=features)
              .sort_values(ascending=False).head(15).reset_index())
importance.columns = ["feature", "avg_impact"]

# 5. Trade-off
bad, good = t["TARGET"] == 1, t["TARGET"] == 0
rows = []
for d in [5, 10, 15, 20, 30, 40]:
    dec = t["p"] >= t["p"].quantile(1 - d / 100)
    rows.append({"decline_pct": d,
                 "defaulters_caught": (dec & bad).sum() / bad.sum(),
                 "good_customers_lost": (dec & good).sum() / good.sum(),
                 "approved_default_rate": t.loc[~dec, "TARGET"].mean()})
tradeoff = pd.DataFrame(rows)

for name, df in {"bands": bands, "deciles": deciles, "segments": segments,
                 "importance": importance, "tradeoff": tradeoff}.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
    print(f"{name:11s} {df.shape}")

bands       (3, 5)
deciles     (10, 4)
segments    (15, 6)
importance  (15, 2)
tradeoff    (6, 4)
